# 11 — Hydro-Geological Collaborative Filtering (HG-CF)
**Matrix Factorization with Graph Laplacian Regularization**

Optimizes latent factor representations regularized by physical hydrogeology:
$$\min_{P, Q} \frac{1}{2}\sum_{(u,i)\in \Omega} (R_{ui} - p_u^T q_i)^2 + \frac{\alpha}{2}\text{Tr}(P^T L_H P) + \frac{\lambda}{2}(\|P\|_F^2 + \|Q\|_F^2)$$
where $L_H = D_H - S_H$ is the Graph Laplacian over the Gaussian RBF hydrogeological similarity kernel $S_{uv}^H$.

In [ ]:
import sys, os, pandas as pd
sys.path.insert(0, os.path.abspath('..'))
from src.data_loading import load_districts_hydrogeology
from src.recommendation import HGCFRecommendationService

districts_df = load_districts_hydrogeology()
profiles = []
for _, r in districts_df.iterrows():
    profiles.append({
        'id': r['district_id'],
        'name': r['district_name'],
        'soil_type': r['major_soil_type'] or 'Alluvial',
        'slope_pct': float(r['average_slope_percent'] or 1.5),
        'rainfall_departure_pct': -5.0,
        'stage_extraction_pct': float(r['stage_of_extraction_pct'] or 75.0),
        'depth_m': 14.0
    })

service = HGCFRecommendationService(n_factors=6, n_iterations=20)
service.fit(profiles)

target_district = profiles[0]['id']
recs = service.recommend(target_district, top_k=4)
print(f'Top Interventions for {profiles[0]["name"]}:')
display(pd.DataFrame(recs)[['name', 'category', 'suitability_score', 'unit_cost_inr', 'evidence_tier']])

# Cold-start on unseen district
cold_recs = service.cold_start_recommend({
    'soil_type': 'Basaltic Black Soil',
    'slope_pct': 2.8,
    'rainfall_departure_pct': -14.0,
    'stage_extraction_pct': 95.0,
    'depth_m': 16.0
}, top_k=3)
print('Cold-Start Recommendations for Unseen Basalt Terrain:')
display(pd.DataFrame(cold_recs)[['name', 'suitability_score', 'evidence_tier']])